# Case Definition 1: At least one occurrence of the breast cancer diagnostic codes

In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "breast_cancer_condition" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_57829836_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4091464, 4091469, 4112853, 4162253, 81250)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_57829836_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_57829836",
  "condition_57829836_*.csv")
message(str_glue('The data will be written to {condition_57829836_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_57829836_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_57829836_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_57829836_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
condition_df <- read_bq_export_from_workspace_bucket(condition_57829836_path)

dim(condition_df)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_57829836/condition_57829836_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_57829836/condition_57829836_000000000000.csv.



[1] 865233      5

In [2]:
unique(condition_df$standard_concept_name)

[1] "Primary malignant neoplasm of male breast"                                   
 [2] "Carcinoma in situ of female breast"                                          
 [3] "Secondary malignant neoplasm of female breast"                               
 [4] "Carcinoma of breast - upper, outer quadrant"                                 
 [5] "Overlapping malignant neoplasm of male breast"                               
 [6] "Primary malignant neoplasm of areola of female breast"                       
 [7] "Infiltrating lobular carcinoma of left female breast"                        
 [8] "Inflammatory carcinoma of breast"                                            
 [9] "Malignant phyllodes tumor of breast"                                         
[10] "Primary malignant neoplasm of female left breast"                            
[11] "Malignant neoplasm of upper-outer quadrant of female breast"                 
[12] "Lobular carcinoma in situ of breast"                                         
[13] "Carcinoma in situ of left breast"                                            
[14] "Malignant neoplasm of breast upper inner quadrant"                           
[15] "Mucinous carcinoma of breast"                                                
[16] "Papillary carcinoma in situ of breast"                                       
[17] "Malignant neoplasm of upper-inner quadrant of female breast"                 
[18] "Carcinoma of female breast"                                                  
[19] "Primary malignant neoplasm of central portion of female breast"              
[20] "Secondary malignant neoplasm of breast"                                      
[21] "Carcinoma of male breast"                                                    
[22] "Hereditary breast and ovarian cancer syndrome"                               
[23] "Malignant melanoma of skin of breast"                                        
[24] "Malignant neoplasm of axillary tail of female breast"                        
[25] "Malignant neoplasm of nipple and areola of male breast"                      
[26] "Lobular carcinoma in situ of left breast"                                    
[27] "Malignant neoplasm, overlapping lesion of breast"                            
[28] "Intraductal carcinoma in situ of bilateral breasts"                          
[29] "Carcinoma in situ of right breast"                                           
[30] "Primary malignant neoplasm of lower inner quadrant of female breast"         
[31] "Malignant neoplasm of lower-outer quadrant of female breast"                 
[32] "Infiltrating duct carcinoma of breast"                                       
[33] "Primary malignant neoplasm of axillary tail of right female breast"          
[34] "Metastatic human epidermal growth factor 2 positive carcinoma of breast"     
[35] "Infiltrating duct carcinoma of right female breast"                          
[36] "Triple-negative breast cancer"                                               
[37] "Primary malignant neoplasm of axillary tail of left female breast"           
[38] "Primary malignant neoplasm of breast upper inner quadrant"                   
[39] "Malignant neoplasm of axillary tail of breast"                               
[40] "Invasive carcinoma of breast"                                                
[41] "Primary malignant neoplasm of upper inner quadrant of female breast"         
[42] "Local recurrence of malignant tumor of breast"                               
[43] "Primary malignant neoplasm of breast"                                        
[44] "Malignant neoplasm of breast upper outer quadrant"                           
[45] "Malignant neoplasm of female breast"                                         
[46] "HER2-positive carcinoma of breast"                                           
[47] "Malignant neoplasm of central part of female breast"                         
[48] "Lobular carcinoma in situ of right breast"     

In [3]:
case_definition1<-unique(condition_df$person_id)
length(case_definition1)

[1] 10712

# Case Definition 2: Personal Health History Survey Indication

In [4]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "bc" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_28116944_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (836772)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_28116944_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_28116944",
  "survey_28116944_*.csv")
message(str_glue('The data will be written to {survey_28116944_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_28116944_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_28116944_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_28116944_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
survey_df <- read_bq_export_from_workspace_bucket(survey_28116944_path)

dim(survey_df)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_28116944/survey_28116944_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_28116944/survey_28116944_000000000000.csv.



[1] 104257      9

In [5]:
bc<-survey_df[survey_df$answer == "Including yourself, who in your family has had breast cancer? - Self",]
case_definition2<-unique(bc$person_id)
length(case_definition2)

[1] 6778

In [6]:
length(case_definition1)
length(case_definition2)
full_list<-c(case_definition1, case_definition2)
cases<-unique(full_list)
length(cases)

[1] 10712

[1] 6778

[1] 12450

# Get Controls

In [7]:
# This snippet assumes that you run setup first

# This code copies a file from your Google Bucket into a dataframe

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- 'Demographic_and_ancestry_covariates.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
demographics  <- read_csv(name_of_file_in_bucket)

character(0)

Rows: 162629 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): SexGender, income, education, where_born, military, healthcare, di...
dbl  (9): person_id, race_unknown, age_today, LGBTQIA, ehr_length, relative_...
lgl  (8): AIAN, Asian, Black, Mid, Multiple, PI, White, His
date (3): date_of_birth, min_date, max_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [8]:
table(demographics$SexGender[demographics$person_id %in% cases])



Cis_female   Cis_male        SGM 
      6687        117        146 

In [9]:
controls <- demographics$person_id[!(demographics$person_id %in% cases)]
table(demographics$SexGender[demographics$person_id %in% controls])
length(controls)


Cis_female   Cis_male        SGM 
     91211      60030       4438 

[1] 155679

In [10]:
#Control for Sex/Gender
# Filter the demographic dataframe to remove rows where SexGender is "Cis_male"
non_male_ids <- demographics %>%
  filter(SexGender != "Cis_male") %>%
  select(person_id)

# Filter the cases dataframe by retaining only rows with person_id in cis_woman_ids
cases <- cases[cases %in% non_male_ids$person_id]
length(cases)

# Filter the controls dataframe by retaining only rows with person_id in cis_woman_ids
controls <- controls[controls %in% non_male_ids$person_id]
length(controls)

[1] 6833

[1] 95649

In [11]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "mastectomy" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_81554391_procedure_sql <- paste("
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4286804)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) procedure 
    LEFT JOIN
        `concept` p_standard_concept 
            ON procedure.procedure_concept_id = p_standard_concept.concept_id 
    LEFT JOIN
        `concept` p_type 
            ON procedure.procedure_type_concept_id = p_type.concept_id 
    LEFT JOIN
        `concept` p_modifier 
            ON procedure.modifier_concept_id = p_modifier.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON procedure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` p_visit 
            ON v.visit_concept_id = p_visit.concept_id 
    LEFT JOIN
        `concept` p_source_concept 
            ON procedure.procedure_source_concept_id = p_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedure_81554391_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedure_81554391",
  "procedure_81554391_*.csv")
message(str_glue('The data will be written to {procedure_81554391_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_81554391_procedure_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  procedure_81554391_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedure_81554391_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), procedure_type_concept_name = col_character(), modifier_concept_name = col_character(), visit_occurrence_concept_name = col_character(), procedure_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), modifier_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
procedure_df <- read_bq_export_from_workspace_bucket(procedure_81554391_path)

unique(procedure_df$standard_concept_name)

head(procedure_df, 5)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_81554391/procedure_81554391_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_81554391/procedure_81554391_000000000000.csv.



[1] "Modified radical mastectomy, bilateral"                                
 [2] "Mastectomy for gynecomastia"                                           
 [3] "Partial mastectomy"                                                    
 [4] "Lumpectomy of breast"                                                  
 [5] "Bilateral subcutaneous mammectomy"                                     
 [6] "Excisional biopsy of breast"                                           
 [7] "Radical mastectomy including pectoral muscles and axillary lymph nodes"
 [8] "Partial mastectomy with axillary lymphadenectomy"                      
 [9] "Modified radical mastectomy"                                           
[10] "Excision of aberrant tissue of breast"                                 
[11] "Lumpectomy of right breast"                                            
[12] "Bilateral mastectomy"                                                  
[13] "Total excision of lactiferous duct of left breast"                     
[14] "Excision of lactiferous duct fistula of breast"                        
[15] "Simple mastectomy"                                                     
[16] "Excisional biopsy of breast mass"                                      
[17] "Lumpectomy of left breast"                                             
[18] "Subcutaneous mastectomy"                                               
[19] "Simple mastectomy of right breast"                                     
[20] "Mastectomy of left breast"                                             
[21] "Bilateral simple mastectomy"                                           
[22] "Bilateral radical mastectomy"                                          
[23] "Re-excision of breast for clearance of tumor margins"                  
[24] "Excision of breast tissue"                                             
[25] "Radical mastectomy"                                                    
[26] "Quadrantectomy of breast"                                              
[27] "Excision of nipple"                                                    
[28] "Wide local excision of breast lesion"                                  
[29] "Prophylactic mastectomy"                                               
[30] "Simple mastectomy of left breast"                                      
[31] "Bilateral subcutaneous mammectomy with synchronous implant"            
[32] "Mastectomy of right breast"                                            
[33] "Bilateral mastectomy extended simple"

person_id,procedure_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,procedure_datetime,procedure_type_concept_id,procedure_type_concept_name,modifier_concept_id,modifier_concept_name,quantity,visit_occurrence_id,visit_occurrence_concept_name,procedure_source_value,procedure_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,modifier_source_value
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
1940776,4061934,"Modified radical mastectomy, bilateral",17086001,SNOMED,2013-06-16 11:11:51.853 UTC,32833,EHR order,NA,NA,NA,NA,NA,17086001,4061934,"Modified radical mastectomy, bilateral",17086001,SNOMED,NA
2248625,4061934,"Modified radical mastectomy, bilateral",17086001,SNOMED,2014-11-02 17:06:21.12 UTC,32833,EHR order,NA,NA,NA,NA,NA,17086001,4061934,"Modified radical mastectomy, bilateral",17086001,SNOMED,NA
3228130,4242107,Mastectomy for gynecomastia,59620004,SNOMED,1988-04-19 06:00:00 UTC,32821,EHR billing record,NA,NA,NA,6.6e+16,Outpatient Visit,19140,42733286,Mastectomy for gynecomastia (Deprecated),19140,CPT4,NA
2942250,4275911,Partial mastectomy,64368001,SNOMED,2021-04-06 11:37:00 UTC,44786630,Primary Procedure,NA,NA,1,9.0e+15,Outpatient Visit,NA,NA,NA,NA,NA,NA
6516723,4242107,Mastectomy for gynecomastia,59620004,SNOMED,2003-01-06 11:45:00 UTC,38000269,Outpatient header - 1st position,0,No matching concept,1,6.4e+16,Outpatient Visit,19140,42733286,Mastectomy for gynecomastia (Deprecated),19140,CPT4,No matching concept


In [12]:
length(controls)
controls <- controls[!controls %in% procedure_df$person_id]
length(controls)

[1] 95649

[1] 95499

In [13]:
df_cases <- data.frame(
  person_id = cases,
  status = 1
)

df_controls <- data.frame(
  person_id = controls,
  status = 0
)

final_df <- rbind(df_cases, df_controls)
nrow(final_df)
n_distinct(final_df$person_id)

[1] 102332

[1] 102332

In [14]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- final_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'eMERGE_breast_cancer_case_control_df.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)

character(0)

[1] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data.csv"                                               
  [2] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data_domain_filtered_60.csv"                            
  [3] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Area_level_disease_statistics.csv"                               
  [4] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_Control_df.csv"                                             
  [5] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_SIRE_cohort.csv"                  
  [6] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_cohort.csv"                       
  [7] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_SIRE_cohort.csv"                   
  [8] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_cohort.csv"                        
  [9] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Demographic_and_ancestry_covariates.csv"                         
 [10] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Afib.csv"                           
 [11] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Asthma.csv"                         
 [12] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_BreastC.csv"                        
 [13] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CHD.csv"                            
 [14] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CKD.csv"                            
 [15] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_HyperC.csv"                         
 [16] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_ProstateC.csv"                      
 [17] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t1d.csv"                            
 [18] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t2d.csv"                            
 [19] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Afib.csv"                          
 [20] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Asthma.csv"                        
 [21] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_BreastC.csv"                       
 [22] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CHD.csv"                           
 [23] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CKD.csv"                           
 [24] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_HyperC.csv"                        
 [25] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_ProstateC.csv"                     
 [26] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t1d.csv"                           
 [27] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t2d.csv"                           
 [28] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Afib.csv"                          
 [29] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Asthma.csv"                        
 [30] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_BreastC.csv"                       
 [31] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_CHD.csv"